# 03 · Synthetic generation + curation (CuratorKIT → SGLang)
**Purpose:** generate yes/no/maybe research QA from unlabelled abstracts with the SGLang-served generator, then gate it (grounding/label check, LLM reward judge, MinHash dedup, cleaning).
First a 30-abstract pilot to pick the generator and tune thresholds (target 30–60 % rejected), then the full N in resumable chunks.
**Inputs:** `artifacts/data/source_abstracts.jsonl`, `data/fewshot.json`. **Outputs:** `artifacts/curator/<run>/chunk_*/` (CuratorKIT native outputs: `manifest.json`, `rejected.jsonl`, exports) + our flat `passed.jsonl`/`rejected.jsonl` per chunk; `data/generator_choice.json`, `data/pilot_runs.json`, `data/curation_stats.json`.
**Runtime (T4, 7B-AWQ):** pilot ~5 min, full 600 abstracts ~45–60 min.

In [ ]:
import os, sys, json, time
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
G = cfg["generation"]; SEED = cfg["seed"]
source = C.read_jsonl(C.ART / "data" / "source_abstracts.jsonl")
fewshot = json.load(open(C.DATA / "fewshot.json"))
print("profile", GPU, "| sources", len(source))

In [ ]:
# Generator: try the profile's first choice, fall back if SGLang cannot serve it on this GPU.
choice_f = C.DATA / "generator_choice.json"
if choice_f.exists() and not FORCE:
    GEN = json.load(open(choice_f))["generator"]
    base = C.start_server(GEN, "gen", P["sglang_args"] + (P["generator_args"] if "AWQ" in GEN else []), mem_fraction=P["mem_fraction"])
else:
    tried = {}
    for GEN in [P["generator"], P.get("generator_fallback")]:
        if GEN is None:
            continue
        try:
            base = C.start_server(GEN, "gen", P["sglang_args"] + (P["generator_args"] if "AWQ" in GEN else []), mem_fraction=P["mem_fraction"])
            break
        except RuntimeError as e:
            tried[GEN] = str(e)[-1500:]
            print(f"[gen] {GEN} failed:\n{tried[GEN]}")
    else:
        raise RuntimeError("no generator could be served")
    json.dump({"generator": GEN, "gpu": GPU, "failed": tried}, open(choice_f, "w"), indent=1)
print("generator:", GEN)
print(C.chat([{"role": "user", "content": "Reply with the single word: ready"}], "gen", max_tokens=5)["text"])

In [ ]:
import litellm
from curatorkit import Curator, CuratorConfig
os.environ["LITELLM_LOG"] = "ERROR"
litellm.suppress_debug_info = True


def to_curator_rows(rows):
    return [{"id": str(r["pubid"]), "instruction": r["question"], "input": r["context_text"], "output": r["long_answer"]} for r in rows]


def dump(samples, path, rejected):
    C.write_jsonl([{"id": s.id, "source_pubid": s.metadata.get("source_sample_id"), "question": s.instruction,
                    "context_text": s.input, "answer": s.output, "task_type": s.task_type,
                    **({"rejecting_step": s.rejecting_step, "rejection_reason": s.rejection_reason} if rejected else {})}
                   for s in samples], path)


def run_chunk(rows, out_dir, hall, rew):
    """One CuratorKIT run; idempotent via our flat dumps."""
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / "passed.jsonl").exists() and not FORCE:
        return
    src = C.write_jsonl(to_curator_rows(rows), out_dir / "source.jsonl")
    res = Curator(CuratorConfig(
        dataset=str(src), format="alpaca", name=out_dir.name,
        llm_model="openai/gen", llm_api_base=base, llm_api_key="none",
        llm_temperature=G["llm_temperature"], llm_max_tokens=G["llm_max_tokens"], llm_extra_body={"seed": SEED},
        llm_concurrency=32, judge_llm_extra_body={"seed": SEED},
        generation_task="qa", num_questions=G["num_questions"], qa_prompt_template=C.qa_prompt_template(fewshot),
        hallucination_threshold=hall, hallucination_prompt_template=C.HALLUCINATION_PROMPT,
        reward_threshold=rew, diversity_threshold=G["diversity_threshold"], dedup=G["dedup"], minhash_seed=SEED, clean=True,
        export_formats=["alpaca", "sharegpt"], output_dir=str(out_dir / "curatorkit"),
    )).run()
    res.print_summary()
    json.dump(res.stage_counts, open(out_dir / "stage_counts.json", "w"), indent=1)
    dump(res.rejected, out_dir / "rejected.jsonl", True)
    dump(res.passed, out_dir / "passed.jsonl", False)   # written last = chunk-complete marker


def summarize(dirs):
    from collections import Counter
    passed = [r for d in dirs for r in C.read_jsonl(d / "passed.jsonl")]
    rejected = [r for d in dirs for r in C.read_jsonl(d / "rejected.jsonl")]
    qa_rej = [r for r in rejected if r["source_pubid"] is not None]
    gen_fail = [r for r in rejected if r["source_pubid"] is None]
    reason = lambda r: r["rejection_reason"].split(":")[0]
    lab = lambda rows: Counter(C.parse_answer(r["answer"]) for r in rows)
    n_gen = len(passed) + len(qa_rej)
    return {"generated_qa": n_gen, "passed": len(passed), "rejected_qa": len(qa_rej),
            "rejection_rate": len(qa_rej) / max(n_gen, 1), "source_level_failures": len(gen_fail),
            "rejected_by_step": dict(Counter(r["rejecting_step"] for r in qa_rej)),
            "rejected_by_reason": dict(Counter(reason(r) for r in qa_rej)),
            "labels_passed": dict(lab(passed)), "labels_rejected": dict(lab(qa_rej))}

## Pilot (30 abstracts) — threshold tuning. Edit `HALL`/`REW` and re-run this cell; every pilot is logged to `data/pilot_runs.json`.

In [ ]:
HALL, REW = G["hallucination_threshold"], G["reward_threshold"]
pilot_dir = C.ART / "curator" / f"pilot_h{HALL}_r{REW}"
t0 = time.time()
run_chunk(source[:G["pilot_n"]], pilot_dir, HALL, REW)
pilot = {"hall": HALL, "reward": REW, "generator": GEN, "wall_s": round(time.time() - t0), **summarize([pilot_dir])}
log_f = C.DATA / "pilot_runs.json"
log = json.load(open(log_f)) if log_f.exists() else []
log = [p for p in log if (p["hall"], p["reward"], p["generator"]) != (HALL, REW, GEN)] + [pilot]
json.dump(log, open(log_f, "w"), indent=1)
print(json.dumps(pilot, indent=1))
for r in C.read_jsonl(pilot_dir / "rejected.jsonl")[:3] + C.read_jsonl(pilot_dir / "passed.jsonl")[:2]:
    print("\n—", r.get("rejection_reason", "PASSED")[:120], "\nQ:", r["question"], "\nA:", r["answer"])

## Full run (N abstracts, 100-abstract chunks, resumable). Thresholds below are the ones chosen from the pilot.

In [ ]:
HALL, REW = G["hallucination_threshold"], G["reward_threshold"]
FULL = C.ART / "curator" / "full"
CHUNK = 100
dirs = []
for i in range(0, len(source), CHUNK):
    d = FULL / f"chunk_{i // CHUNK:02d}"
    t0 = time.time()
    run_chunk(source[i:i + CHUNK], d, HALL, REW)
    dirs.append(d)
    print(f"chunk {i // CHUNK} done in {time.time() - t0:.0f}s")
C.stop_server()

In [ ]:
stats = {"generator": GEN, "hall_threshold": HALL, "reward_threshold": REW, "n_source": len(source), **summarize(dirs)}
json.dump(stats, open(C.DATA / "curation_stats.json", "w"), indent=1)
print(json.dumps(stats, indent=1))
print("Saved:", FULL, C.DATA / "curation_stats.json")